In [0]:
# records count validation
bronze_count = spark.table(
    "workspace.student_bronze.raw_student_exams"
).count()

silver_count = spark.table(
    "workspace.student_silver.student_exams_clean"
).count()

fact_count = spark.table(
    "workspace.student_gold.fact_exam_performance"
).count()

assert bronze_count > 0, "Bronze table is empty"

assert silver_count > 0, "Silver table is empty"

assert bronze_count >= silver_count, \
    "Unexpected number of Silver records"

assert silver_count == fact_count, \
    "Silver and Gold row count mismatch"

print(f"Bronze records: {bronze_count:,}")
print(f"Silver records: {silver_count:,}")
print(f"Gold records: {fact_count:,}")

print("Row count validation: PASSED")

Bronze records: 100,000
Silver records: 100,000
Gold records: 100,000
Row count validation: PASSED


In [0]:
# Referential integrity validation
result = spark.sql("""
    SELECT
        COUNT(*) AS joined_rows,

        COUNT(DISTINCT f.exam_performance_key)
            AS unique_fact_keys,

        COUNT_IF(s.student_key IS NULL)
            AS missing_student_keys,

        COUNT_IF(st.study_profile_key IS NULL)
            AS missing_study_keys,

        COUNT_IF(l.lifestyle_key IS NULL)
            AS missing_lifestyle_keys,

        COUNT_IF(d.digital_key IS NULL)
            AS missing_digital_keys

    FROM workspace.student_gold.fact_exam_performance f

    LEFT JOIN workspace.student_gold.dim_student s
        ON f.student_key = s.student_key

    LEFT JOIN workspace.student_gold.dim_study_profile st
        ON f.study_profile_key = st.study_profile_key

    LEFT JOIN workspace.student_gold.dim_lifestyle_profile l
        ON f.lifestyle_key = l.lifestyle_key

    LEFT JOIN workspace.student_gold.dim_digital_access d
        ON f.digital_key = d.digital_key
""").first()

assert result["joined_rows"] == fact_count

assert result["unique_fact_keys"] == fact_count

assert result["missing_student_keys"] == 0
assert result["missing_study_keys"] == 0
assert result["missing_lifestyle_keys"] == 0
assert result["missing_digital_keys"] == 0

print("Referential integrity validation: PASSED")

Referential integrity validation: PASSED


In [0]:

# analitycal views validation
views = [
    "agg_academic_performance",
    "agg_study_habits",
    "agg_lifestyle_impact",
    "agg_socioeconomic_factors",
    "agg_digital_learning",
    "agg_exam_preparation",
    "agg_grade_distribution"
]

query = " UNION ALL ".join(
    f"""
    SELECT
        '{view}' AS view_name,
        SUM(student_count) AS total_students
    FROM workspace.student_gold.{view}
    """
    for view in views
)

results = spark.sql(query).collect()

for row in results:

    assert row["total_students"] == fact_count, \
        f"Validation failed: {row['view_name']}"

    print(f"{row['view_name']}: PASSED")

print("Aggregation validation: PASSED")

agg_academic_performance: PASSED
agg_study_habits: PASSED
agg_lifestyle_impact: PASSED
agg_socioeconomic_factors: PASSED
agg_digital_learning: PASSED
agg_exam_preparation: PASSED
agg_grade_distribution: PASSED
Aggregation validation: PASSED


In [0]:
print("=" * 50)
print("STUDENT PERFORMANCE DATA PIPELINE")
print("=" * 50)

print("Row count validation: PASSED")
print("Referential integrity: PASSED")
print("Aggregation validation: PASSED")

print("=" * 50)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 50)

STUDENT PERFORMANCE DATA PIPELINE
Row count validation: PASSED
Referential integrity: PASSED
Aggregation validation: PASSED
PIPELINE COMPLETED SUCCESSFULLY
